In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
import matplotlib.pyplot as plt

In [128]:
#load in results of AI detection
governor = pd.read_csv('text/results/governor_text_detection.csv')
ag = pd.read_csv('text/results/ag_text_detection.csv')
mayor = pd.read_csv('text/results/mayor_text_detection.csv')
us_senate = pd.read_csv('text/results/us_senate_text_detection.csv')

Cleaning functions:

In [130]:
#drop rows where AI assistance score is 'unable to run' or NaN
#get token length and add it to a new column in the given table, necessary for magnitude calculation
def clean_df(df):
    df_1 = df[df['assistance_score'] != 'unable to run']
    df_2 = df_1.dropna(subset=['assistance_score'])
    return df_2

def add_token_length(df):
    df['token_length'] = df['sampled_text'].apply(lambda x: len(nltk.word_tokenize(str(x))))
    return df

In [79]:
governor_cleaned = clean_df(governor)
gov_tokens = add_token_length(governor_cleaned)

In [85]:
ag_cleaned = clean_df(ag)
ag_tokens = add_token_length(ag_cleaned)

In [91]:
mayor_cleaned = clean_df(mayor)
mayor_tokens = add_token_length(mayor_cleaned)

In [132]:
us_senate_cleaned = clean_df(us_senate)
us_senate_tokens = add_token_length(us_senate_cleaned)

In [134]:
#combine all for analyses that are across all races
combined_40 = pd.concat([gov_tokens, ag_tokens, mayor_tokens, us_senate_tokens], ignore_index=True)
combined_40

,content_id,candidate_id,candidate_name,page_url,page_type,link_type,unprocessed_text,cleaned_text,sampled_text,race_type,year,AI_label,assistance_score,confidence,fraction_ai,fraction_ai_assisted,fraction_human,num_ai_segments,token_length
0,1,6179,Andy Beshear,https://andybeshear.com/,home,campaign_site,Andy Beshear Menu Meet Andy Meet Jacqueline An...,Andy Beshear Menu Meet Andy Meet Jacqueline An...,Andy Beshear Menu Meet Andy Meet Jacqueline An...,Governor,2023,Human,0.00432,High,0,0,1,0,84
1,2,6180,Daniel Cameron,https://cameronforkentucky.com/,home,campaign_site,Menu News Facebook Twitter Instagram Donate › ...,Menu News Facebook Twitter Instagram Donate S...,I'm In! QUICK DONATE $25 $50 $100 ENTER AMOUNT...,Governor,2023,Human,0.006618,High,0,0,1,0,235
2,3,6180,Daniel Cameron,https://cameronforkentucky.com/policies/terms,policy,campaign_site,Menu News Facebook Twitter Instagram Donate › ...,Menu News Facebook Twitter Instagram Donate S...,Data Privacy We will not share or sell your pe...,Governor,2023,AI,0.968517,High,1,0,0,1,299
3,4,6180,Daniel Cameron,https://cameronforkentucky.com/policies/privacy,policy,campaign_site,Menu News Facebook Twitter Instagram Donate › ...,Menu News Facebook Twitter Instagram Donate S...,"This Privacy Policy explains how we collect, u...",Governor,2023,AI,0.989806,High,1,0,0,1,439
4,5,6180,Daniel Cameron,https://cameronforkentucky.com/issues/,policy,campaign_site,Menu News Facebook Twitter Instagram Donate › ...,Menu News Facebook Twitter Instagram Donate S...,He will always oppose the radical green agenda...,Governor,2023,Human,0.413824,Low,0,0,1,0,290
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
526,2828,6155,Tammy Baldwin,https://www.tammybaldwin.com/about/,about,campaign_site,About Tammy News Issues Get Involved Store Con...,About Tammy News Issues Get Involved Store Con...,"James Allen Veteran Vision Equity Act, passed ...",US Senate,2024,Human,0.004686219617724419,High,0.0,0.0,1.0,0,436
527,2829,6158,Glenn Elliott,https://www.elliottforwv.com,home,campaign_site,Elliott Wave Forecast Login Start 14-Day Trial...,Elliott Wave Forecast Login Start 14-Day Trial...,Retail Traders & Investors Join us never trade...,US Senate,2024,Human,0.10337324101816525,Medium,0.0,0.0,1.0,0,440
528,2830,6157,Jim Justice,http://jimjusticewv.com,home,campaign_site,Skip to content Endorsed by Trump Meet Jim Acc...,Skip to content Endorsed by Trump Meet Jim Acc...,Message frequency may vary. Reply stop to stop...,US Senate,2024,Human,0.004607409238815308,High,0.0,0.0,1.0,0,280
529,2831,6157,Jim Justice,https://jimjusticewv.com/issues/,policy,campaign_site,Skip to content Endorsed by Trump Meet Jim Acc...,Skip to content Endorsed by Trump Meet Jim Acc...,Reply stop to stop and help for help. Mobile o...,US Senate,2024,Human,0.004374942742288113,High,0.0,0.0,1.0,0,54


Mean detection probability (needs to be expanded for the other two content types soon, but for now only works for the text)

In [ ]:
def mean_detection_probability(table, candidate):
    #filter for the candidate
    given_rows = table[table['candidate_name'] == candidate]
    ai_assistance_scores= given_rows['AI assistance score']
    token_lengths = given_rows['len_tokens']

    numerator = sum(token_lengths * ai_assistance_scores)
    denom = sum(token_lengths)
    m_prob = numerator / denom
    
    return m_prob

# Testing differences in samples of 40%, 60%, and 80% test length #

**40% sampling**

In [118]:
combined_40['AI_label'].value_counts(normalize=True) * 100

AI_label
Human    94.915254
AI        4.331450
Mixed     0.753296
Name: proportion, dtype: float64

In [146]:
combined_40['assistance_score'] = combined_40['assistance_score'].astype(float)

In [148]:
np.mean(combined_40['assistance_score'])

0.07220251499782691

**60% sampling**

In [156]:
combined_60_raw = pd.read_csv("text/results/combined_60.csv")

In [158]:
combined_60_cleaned = clean_df(combined_60_raw)
combined_60 = add_token_length(combined_60_cleaned)

In [162]:
combined_60['AI_label'].value_counts(normalize=True) * 100

AI_label
Human    94.727273
AI        4.000000
Mixed     1.272727
Name: proportion, dtype: float64

In [164]:
combined_60['assistance_score'] = combined_60['assistance_score'].astype(float)

In [166]:
np.mean(combined_60['assistance_score'])

0.07090409058370932

**80% sampling**

In [173]:
combined_80_raw = pd.read_csv("text/results/combined_80.csv")

In [175]:
combined_80_cleaned = clean_df(combined_80_raw)
combined_80 = add_token_length(combined_80_cleaned)

In [177]:
combined_80['AI_label'].value_counts(normalize=True) * 100

AI_label
Human    95.264117
AI        3.096539
Mixed     1.639344
Name: proportion, dtype: float64

In [179]:
combined_60['assistance_score'] = combined_60['assistance_score'].astype(float)
np.mean(combined_60['assistance_score'])

0.07090409058370932

**100% of the text**

In [182]:
combined_100_raw = pd.read_csv("text/results/combined_100.csv")

In [184]:
combined_100_cleaned = clean_df(combined_100_raw)
combined_100 = add_token_length(combined_100_cleaned)

In [188]:
combined_100['AI_label'].value_counts(normalize=True) * 100

AI_label
Human    96.174863
AI        2.003643
Mixed     1.821494
Name: proportion, dtype: float64

In [240]:
archiving = pd.read_csv("/Users/agueorg/Desktop/WeberLab/anna-RA/candidate-scraping/politicalemails_results.csv")



In [242]:
archiving['message_count'] = archiving['message_count'].str.replace('+', '', regex=False)
archiving = archiving[~archiving['message_count'].str.contains('error', case=False, na=False)]
archiving['message_count'] = archiving['message_count'].astype(float)

In [244]:
len(archiving[archiving['message_count'] != 0]) / len(archiving)

0.5110650069156293

In [246]:
non_zero = archiving[archiving['message_count'] != 0]

In [248]:
np.mean(non_zero['message_count'])

33.78213802435724